# 01 — Exploratory Data Analysis & Preprocessing Justification (U0.8)

This notebook presents the dataset anatomy, class balance, object size distribution, resolution shift between train/val and test, ignore-region statistics, and the frame-step redundancy analysis justifying $k=5$.

In [ ]:
%load_ext autoreload
%autoreload 2
from vdet.paths import P
from vdet.data.manifest import load_boxes, load_ignore, load_manifest
from vdet.data.redundancy import generate_all_eda_figures

## 1. Split Summary & Figures Generation

Each figure below maps a concrete empirical observation to a downstream engineering decision (input size, anchor scales, downscale-biased augmentation, and frame subsampling step $k=5$).

In [ ]:
saved_figs, lag_df = generate_all_eda_figures(cache_root=P.cache, images_dir=P.images)
display(lag_df.head(10))

## 2. Observations & Downstream Decisions

1. **Class Imbalance & Train/Val Domain Gap (`eda_class_distribution.png`, `eda_clip_class_heatmap.png`):** `car` (45.7% train vs 27.9% val) and `person` (29.7% train vs 44.1% val) dominate, while `bus` (0.9% train, 0.2% val) and `truck` (2.8% train, 1.2% val) are rare $\rightarrow$ report per-class AP alongside box counts.
2. **Small Object Dominance at 1280 Scale (`eda_object_sizes.png`, `eda_box_wh_scatter.png`):** At `imgsz=1280`, the median box is ~30 px and p10 is ~15 px $\rightarrow$ default COCO 32 px smallest anchors are too coarse; fit k-means anchors (F2) and evaluate high-resolution `imgsz=1600` / P2 heads (F3).
3. **High Per-Frame Density (`eda_density_per_clip.png`):** Sequences reach up to 180 objects/frame in train and 116 in val $\rightarrow$ `maxDets=500` is mandatory for evaluation and inference, and RT-DETR queries are scaled from 300 to 600 (F4).
4. **Test Resolution Shift (`eda_resolutions.png`):** ~16% of test frames are downscaled to $680\times 382$ or $960\times 540$ $\rightarrow$ bias training scale augmentation toward random downscaling ($[0.5, 1.0]$) and apply tiling/upscaling at inference (M4).
5. **Frame-Step Redundancy (`eda_frame_step_redundancy.png`):** Adjacent frames ($k=1$) have median same-track IoU $>0.85$; at $k=5$, displacement reaches ~0.5 box diagonals while retaining 4,868 diverse training frames within the 1.5 GPU-h P0 budget.